In [1]:
pip install gtfs-realtime-bindings

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from google.transit import gtfs_realtime_pb2
import time

# 1. Your API Key (Inserted for you)
API_KEY = "h7i00LqgTiaaReuCvKy6TndN4xJrMR1y"

# 2. The Official URL for Live Bus Positions
URL = f"https://otd.delhi.gov.in/api/realtime/VehiclePositions.pb?key={API_KEY}"

print("📡 Connecting to Delhi Government Server...")

try:
    # 3. Request the data
    response = requests.get(URL)
    
    # Check if the server accepted our key (Status Code 200 = OK)
    if response.status_code == 200:
        print("✅ SUCCESS! Key is valid. Access granted.")
        
        # 4. Parse the binary data (Protocol Buffers)
        feed = gtfs_realtime_pb2.FeedMessage()
        feed.ParseFromString(response.content)
        
        # 5. Count the buses
        bus_count = len(feed.entity)
        print(f"🚌 Found {bus_count} buses active on the roads right now.")
        
        # 6. Show the first 3 buses as an example of the "Labels"
        if bus_count > 0:
            print("\n--- SAMPLE DATA (What we will analyze) ---")
            for i in range(min(3, bus_count)): # Show max 3 buses
                bus = feed.entity[i]
                print(f"\nBus #{i+1}:")
                # Try to get data, handle cases where some fields might be empty
                if bus.HasField('trip_update'):
                    print(f"  - Trip ID: {bus.trip_update.trip.trip_id}")
                    print(f"  - Route ID: {bus.trip_update.trip.route_id}")
                
                if bus.HasField('vehicle'):
                    print(f"  - Latitude: {bus.vehicle.position.latitude}")
                    print(f"  - Longitude: {bus.vehicle.position.longitude}")
                    print(f"  - Speed: {bus.vehicle.position.speed} m/s")
        else:
            print("⚠️ Connected, but no buses found. (Is it late night?)")
            
    elif response.status_code == 401:
        print("❌ ERROR: Unauthorized. The API Key might be wrong.")
    else:
        print(f"❌ ERROR: Server returned code {response.status_code}")

except Exception as e:
    print(f"❌ CONNECTION ERROR: {e}")

📡 Connecting to Delhi Government Server...
✅ SUCCESS! Key is valid. Access granted.
🚌 Found 2737 buses active on the roads right now.

--- SAMPLE DATA (What we will analyze) ---

Bus #1:
  - Latitude: 28.523740768432617
  - Longitude: 77.25592803955078
  - Speed: 0.0 m/s

Bus #2:
  - Latitude: 28.729169845581055
  - Longitude: 77.12256622314453
  - Speed: 0.0 m/s

Bus #3:
  - Latitude: 28.660930633544922
  - Longitude: 77.18372344970703
  - Speed: 0.0 m/s


In [3]:
import requests
from google.transit import gtfs_realtime_pb2

API_KEY = "h7i00LqgTiaaReuCvKy6TndN4xJrMR1y"

# We will test two different URL endpoints
endpoints = {
    "Vehicle Positions": f"https://otd.delhi.gov.in/api/realtime/VehiclePositions.pb?key={API_KEY}",
    "Trip Updates": f"https://otd.delhi.gov.in/api/realtime/TripUpdates.pb?key={API_KEY}"
}

print(f"📡 Testing OTD API connectivity...")

for name, url in endpoints.items():
    print(f"\n--- Checking {name} ---")
    try:
        response = requests.get(url)
        if response.status_code == 200:
            feed = gtfs_realtime_pb2.FeedMessage()
            feed.ParseFromString(response.content)
            count = len(feed.entity)
            print(f"✅ Status: ONLINE")
            print(f"📊 Data Count: {count} entries found.")
        else:
            print(f"❌ Status: ERROR {response.status_code}")
    except Exception as e:
        print(f"❌ Connection Failed: {e}")

print("\n--------------------------------")
print("VERDICT:")
print("If you see 'Status: ONLINE' (even with 0 data), your Key is perfect.")
print("If data is 0, the govt server is just having a nap. Try again in 1 hour.")

📡 Testing OTD API connectivity...

--- Checking Vehicle Positions ---
✅ Status: ONLINE
📊 Data Count: 2737 entries found.

--- Checking Trip Updates ---
❌ Status: ERROR 404

--------------------------------
VERDICT:
If you see 'Status: ONLINE' (even with 0 data), your Key is perfect.
If data is 0, the govt server is just having a nap. Try again in 1 hour.


In [4]:
import requests
from google.transit import gtfs_realtime_pb2

# --- CONFIGURATION ---
API_KEY = "h7i00LqgTiaaReuCvKy6TndN4xJrMR1y"
URL = f"https://otd.delhi.gov.in/api/realtime/VehiclePositions.pb?key={API_KEY}"

print("🔎 TESTING API KEY...")

try:
    # 1. Ping the Server
    response = requests.get(URL, timeout=10)
    
    # 2. Check Result
    if response.status_code == 200:
        feed = gtfs_realtime_pb2.FeedMessage()
        feed.ParseFromString(response.content)
        
        bus_count = 0
        for entity in feed.entity:
            if entity.HasField('vehicle'):
                bus_count += 1
        
        print(f"✅ SUCCESS: Connected to OTD Server.")
        print(f"🚌 ACTIVE BUSES FOUND: {bus_count}")
        print("Ready for launch.")
    else:
        print(f"❌ FAILURE: Server returned code {response.status_code}")
        print("Check your internet or the API Key.")

except Exception as e:
    print(f"❌ ERROR: {e}")

🔎 TESTING API KEY...
✅ SUCCESS: Connected to OTD Server.
🚌 ACTIVE BUSES FOUND: 2737
Ready for launch.
